### Assuming the input ISP 2024 trace data is downloaded and stored locally first. This notebook helps to further process data into four different scenarios with Hydro data.
* This data is then fed into the Pras simulation Julia notebook
* For larger files, more than 50MB, are not stored in GitHub because of the limitation. Users can pre-process it locally.

In [157]:
import sys
import os
import ast
import glob
import calendar
import warnings
import numpy as np 
from sklearn.cluster import KMeans
import geopandas as gpd
from shapely.geometry import MultiPoint

import pandas as pd
import matplotlib.pyplot as plt
import time
import datetime as datetime
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None 
# Adjust precision as needed
pd.set_option('display.float_format', '{:.10f}'.format) 

area_code_region = {1:"NNSW", 2:"CNSW", 3:"SNW", 4:"SNSW", 5:"WNV",
                        6:"MEL", 7:"SEV",  8:"NQ", 9:"CQ", 10:"GG",
                        11:"SQ", 12:"NSA", 13:"CSA", 14:"SESA", 15:"TAS"}
region_map = {"NNSW":"NSW", "CNSW":"NSW", "SNW":"NSW", "SNSW":"NSW", "WNV":"VIC",
                    "MEL":"VIC", "SEV":"VIC",  "NQ":"QLD", "CQ":"QLD", "GG":"QLD",
                    "SQ":"QLD", "NSA":"SA", "CSA":"SA", "SESA":"SA", "TAS":"TAS"}

In [ ]:
# The following process depends on prior downloading of ISP 2024 Traces as per the file path - scenario_dem_path and
# scenario_ren_path
def load_all_scenario_data():
    all_scenario_post_24 = {}
    scenario = ['2024_ISP_Progressive_Change', '2024_ISP_Step_Change', '2024_ISP_Green_Energy_Exports']
    start_date = '2024-07-01 00:00:00'
    year =  start_date.split('-')[0]
    scenario_data_dict = {}
    for scene in scenario:
        scenario_dem_path = 'data/ISP_data/2024_ISP_Model/' + scene + '/Traces/demand'
        scenario_ren_path = 'data/ISP_data/2024_ISP_Model/' + scene + '/Traces/load_subtractor'
        DEMAND_DATA_DIR =  os.path.join(os.getcwd(), '../../../../', scenario_dem_path)
        REN_DATA_DIR =  os.path.join(os.getcwd(), '../../../../', scenario_ren_path)
        scene_key =  scene + '_demand'
        scenario_data_dict[scene_key] = load_demand_data(start_date, DEMAND_DATA_DIR)
        merged_dem_cols = pd.concat(scenario_data_dict[scene_key].values(), axis=1)
        #merged_dem_cols = zone_sum_demand(merged_dem_cols, ISP_date="2024")
        scene_key = scene + '_gen'
        ren_data = load_renewable_data(start_date, REN_DATA_DIR)
        merged_ren_cols = pd.concat(ren_data.values(), axis=1)
        
        all_scenario_post_24[scene] = pd.concat([merged_dem_cols, merged_ren_cols], axis=1)
    
    return all_scenario_post_24


all_scenario_post_24 = load_all_scenario_data()
fl_path = os.path.join(os.getcwd(), "../../../../data", "ISP_data", "output")
for k, v in all_scenario_post_24.items():
    fl_name = fl_path + k + ".parquet"
    v.index.name = 'timestamp'
    v.to_parquet(fl_name, index=True)

step_change_All = pd.read_parquet(os.path.join(fl_path, "2024_ISP_Step_Change.parquet"))
step_change = step_change_All[step_change_All.index < '2044-07-01']
step_change.to_csv(os.path.join(fl_path, "2024_ISP_Step_Change_20yrs.csv"))


In [2]:
fl_path = os.path.join(os.getcwd(), "../../../../data", "ISP_data", "output")

multi_year_data = pd.read_csv(os.path.join(fl_path, "2024_ISP_Step_Change_20yrs.csv"))
multi_year_data['timestamp'] = pd.to_datetime(multi_year_data['timestamp'])
multi_year_data = multi_year_data.set_index('timestamp')

In [26]:
def load_gen_data():
    lg = pd.read_csv("../data/output/acdc_load_gen_bus.csv")
    lg['area'] = lg['area'].map(area_code_region)
    fuel_dict= {0:"load", 4:"gas", 8:"coal",16:"hydro",17:"liquid_fuel",20:"biomass",21:"solar",22:"wind"}
    lg['fuel'] = lg['prime_mover_type'].map(fuel_dict)
    reg_dict = {'NNSW':'NSW','CNSW':'NSW','SNW':'NSW','SNSW':'NSW',
                "WNV":"VIC","MEL":"VIC","SEV":"VIC",
                "NQ":"QLD","CQ":"QLD","SQ":"QLD","GG":"QLD",
               "NSA":"SA","CSA":"SA","SESA":"SA","TAS":"TAS"}
    lg['region'] = lg["area"].map(reg_dict)
    return lg
    
def get_scaled_data(myd, region_map):
    min_base = pd.DataFrame(myd.resample('M').max().min()).reset_index(names='profile')
    min_base = min_base.rename(columns={0:'min_base'})
    scaling = pd.DataFrame(myd.resample('M').max().max()/myd.resample('M').max().min()).reset_index(names='profile')
    scaling = scaling.rename(columns={0:'scaling'})
    min_base = min_base.merge(scaling, on='profile')
    min_base[['area', 'fuel']] = min_base['profile'].str.split('_', expand=True)
    lg = load_gen_data()
    xx = lg[['area', 'region']].drop_duplicates().reset_index()
    #min_base = min_base.merge(xx, on='area')
    min_base["region"] = min_base.area.map(region_map)
    min_base.loc[min_base.region.isnull(), "region"] = min_base["area"]
    pl = lg[lg.comp_type == 'PowerLoad']
    pl = pl[['area', 'region', 'active_power']].groupby(['area', 'region']).sum('active_power').reset_index()
    #pp = pl.groupby('region').sum('active_power').reset_index(names="total_power")
    pl['fuel'] =  len(pl) * ["demand"]
    min_base = min_base.merge(pl[['area', 'fuel', 'active_power']], on=['area', 'fuel'], how='left')
    min_base['fuel'] = min_base['fuel'].str.lower()
    rg = lg[lg.comp_type != 'PowerLoad']
    rg = rg[rg.comp_type != 'ThermalStandard']
    rg = rg[(rg.fuel != 'hydro') & (rg.fuel != 'biomass')]
    af = rg.groupby(['area', 'fuel'])['rating'].sum('rating').reset_index()
    #rg.merge(lg['area
    #rg = rg.merge(lg[['area', 'region']], on='area')
    #af = af.merge(lg[['area', 'region']], on='area')
    
    xx = lg[['area', 'region']].drop_duplicates()
    af = af.merge(xx, on='area')
    total_gen = af.groupby(['region', 'fuel'])['rating'].sum('rating').reset_index(name='total_rating')
    af = af.merge(total_gen, on = ['region', 'fuel'])
    min_base = min_base.merge(af[["region",	"fuel", "total_rating"]], on=['region', 'fuel'], how='left')
    min_base = min_base.drop_duplicates()
    vic_value = pl[pl.region == "VIC"]['active_power'].sum()
    min_base.loc[min_base.profile == "VIC_demand", "active_power"] = vic_value
    min_base.loc[min_base.total_rating.isnull(), "total_rating"] = min_base['active_power']
    min_base["new_scaling"] = (min_base["min_base"] / min_base["total_rating"]) * min_base["scaling"]
    my_scaling_dict = dict(zip(min_base['profile'], min_base['new_scaling']))
    return my_scaling_dict

def rescale_data_1_year(demand):
    transformed_data = {}
    new_value = 1
    amin = demand.min()
    amin = pd.DataFrame(amin).T
    amin.replace(0, new_value, inplace=True)
    amin = amin.min()
    maxv = demand.max()/amin
    maxvv = maxv.abs()
    maxvv[maxvv > 1.5] = 1.5  
    for k in demand.columns.to_list():  
        minv = 1
        if "Solar" in k or "Wind" in k:
            minv = 0
            maxvv[k] = 1.5
        if maxv[k] < 0:
            minv = -1
        
        scaler_custom = MinMaxScaler(feature_range=(minv, maxvv[k]))
        xx=demand[k].to_list()
        new_list = [[item] for item in xx]
        scaled_data_custom = scaler_custom.fit_transform(new_list) 
        transformed_data[k] = pd.DataFrame(scaled_data_custom)[0].to_list()
    transformed_data = pd.DataFrame(transformed_data)
    return transformed_data

def rescale_data_20_years(demand, my_scaling_dict):
    minv = 1
    transformed_data = {}
    for k in demand.columns.to_list():
        minv = 1
        if "Solar" in k or "Wind" in k or minv > my_scaling_dict[k]:
            minv = 0
        scaler_custom = MinMaxScaler(feature_range=(minv, my_scaling_dict[k]))
        xx=demand[k].to_list()
        new_list = [[item] for item in xx]
        scaled_data_custom = scaler_custom.fit_transform(new_list) 
        transformed_data[k] = pd.DataFrame(scaled_data_custom)[0].to_list()
    transformed_data = pd.DataFrame(transformed_data)
    return transformed_data

def load_hydro_data():
    
    hydro_path = os.path.join(os.getcwd(), '../../../../data', 'ISP_data','2024_ISP_Model',
                              '2024_ISP_Step_Change','Traces', 'hydro')
    hydro_files = glob.glob(hydro_path + "/Monthly*.csv")
    hydro_dict = {}
    for fl in hydro_files:
        key = fl.split("/")[-1]
        key = key.split(".")[0]
        df = pd.read_csv(fl)
        df['Year'] = df['Year'].astype(str)
        df['Month'] = df['Month'].astype(str).str.zfill(2) # Pad month with leading zero if needed
        df['Day'] = df['Day'].astype(str).str.zfill(2)
    
        df['timestamp'] = df['Year'] + '-' + df['Month'] + '-' + df['Day']
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.set_index('timestamp')
    #df['timestamp'].reset_index(inplace=True)
    #df = df[df.index < '2025-07-01']
    
        hydro_dict[key] = df['Inflows']
    return hydro_dict

In [33]:
odir = os.path.join(os.getcwd(), "data",  "output")
myd = multi_year_data[multi_year_data.index < '2025-07-01']

transformed_data = rescale_data_1_year(myd)
transformed_data = transformed_data.set_index(myd.index)

postfix = "_norm"
all_transformed = transformed_data.add_suffix(postfix)
all_transformed[all_transformed < 0] = 0

all_transformed.to_csv(os.path.join(odir, "2024_ISP_Step_Change_1yr_scaled.csv"))

myd = multi_year_data[multi_year_data.index < '2030-07-01']
my_scaling_dict = get_scaled_data(myd, region_map)

all_transformed = rescale_data_20_years(myd, my_scaling_dict)
postfix = "_norm"
all_transformed = all_transformed.add_suffix(postfix)
all_transformed = all_transformed.set_index(myd.index)
all_transformed[all_transformed < 0] = 0
all_transformed.to_csv(os.path.join(odir, "2024_ISP_Step_Change_6yrs_scaled.csv"))

myd = multi_year_data[multi_year_data.index < '2044-07-01']
my_scaling_dict = get_scaled_data(myd, region_map)
all_transformed = rescale_data_20_years(myd, my_scaling_dict)
postfix = "_norm"
all_transformed = all_transformed.add_suffix(postfix)
all_transformed = all_transformed.set_index(myd.index)
all_transformed[all_transformed < 0] = 0
all_transformed.to_parquet(os.path.join(odir, "2024_ISP_Step_Change_20yrs_scaled.pq"))
#all_transformed.to_csv(os.path.join(odir, "2024_ISP_Step_Change_20yrs_scaled.csv"))


In [34]:
myd = multi_year_data[multi_year_data.index < '2030-07-01']
my_scaling_dict = get_scaled_data(myd, region_map)

all_transformed = rescale_data_20_years(myd, my_scaling_dict)
postfix = "_norm"
all_transformed = all_transformed.add_suffix(postfix)
all_transformed = all_transformed.set_index(myd.index)
all_transformed[all_transformed < 0] = 0

In [215]:
df = all_transformed.copy()
summer_df = df[df.index.month.isin([12, 1, 2])]
daily_max = summer_df.resample('D').agg(['max', 'idxmax']).dropna()
daily_max  = daily_max.filter(like='idxmax', axis='columns')
daily_max.columns = daily_max.columns.droplevel(1)

In [268]:
df = all_transformed.copy()
summer_df = df[df.index.month.isin([12, 1, 2])]
daily_summer_peaks = summer_df.resample('D').max()
daily_summer_peaks = daily_summer_peaks.dropna(axis=0, how='all')
daily_summer_peaks = daily_summer_peaks.reset_index()

daily_summer_peaks['Summer_Year'] = daily_summer_peaks.timestamp.apply(
    lambda x: x.year + 1 if x.month >= 12 else x.year
)


new_index = pd.date_range(start=daily_summer_peaks.timestamp.min(), 
                          end=daily_summer_peaks.timestamp.max(), freq='3D')

new_index = new_index[0:daily_summer_peaks.shape[0]]
#new_index = new_index.floo('D')
daily_summer_peaks = daily_summer_peaks.rename(columns={"timestamp": "original_daily_timestamp"})
daily_summer_peaks["timestamp"] = new_index
daily_summer_peaks = daily_summer_peaks.set_index('timestamp')
daily_summer_peaks.to_csv(os.path.join(odir, "summer_daily_ED_scaled_2025_2030.csv"))

In [270]:
daily_summer_peaks

,original_daily_timestamp,CNSW_demand_norm,CQ_demand_norm,CSA_demand_norm,GG_demand_norm,NNSW_demand_norm,NQ_demand_norm,SESA_demand_norm,SNSW_demand_norm,SNW_demand_norm,...,NSW_Solar_norm,QLD_Solar_norm,SA_Solar_norm,VIC_Solar_norm,NSW_Wind_norm,QLD_Wind_norm,SA_Wind_norm,TAS_Wind_norm,VIC_Wind_norm,Summer_Year
timestamp,,,,,,,,,,,,,,,,,,,,,
2024-12-01,2024-12-01,0.1128028307,0.3527353599,3.7231734210,1.9399692177,0.4653404406,1.1682880231,0.1679539492,1.1874343437,2.2933855216,...,1.0511760088,1.2057768662,0.9910186329,1.3300832216,0.7581683567,0.4343241749,0.7335625152,0.8066753020,0.8180846939,2025
2024-12-04,2024-12-02,0.1206822732,0.3141235679,3.8053792104,1.8306721451,0.4794056853,1.1532552167,0.1755842788,1.1955762277,2.3707941692,...,1.0584337528,1.2175266141,0.9700648728,1.3244183667,0.6354993824,0.3356076155,0.3798247390,0.8127321907,0.8210778586,2025
2024-12-07,2024-12-03,0.1069252314,0.2787741665,3.8139151484,1.7306080148,0.4611602024,1.1394921506,0.1763768128,1.1838924537,2.2305532774,...,1.0545462481,0.9242155292,0.9819664465,1.3278860001,0.6083462994,0.3820316308,0.3990607895,0.6978637645,0.3516856940,2025
2024-12-10,2024-12-04,0.1037900698,0.2651647515,4.4376501265,1.6920843041,0.4567817877,1.1341935698,0.2361651186,1.1811693015,2.1986267593,...,1.0389265621,1.1438884594,0.9576813078,1.3413622732,0.6062508272,0.5416190134,0.4578798341,0.2552277383,0.2850081541,2025
2024-12-13,2024-12-05,0.1110318106,0.2491469279,5.1136881750,1.6467315177,0.4677130331,1.1279546368,0.3024339227,1.1876830044,2.2722371822,...,1.0421373423,0.8939374102,0.9479747730,1.3420388203,0.5689447862,0.5419458642,0.4694040816,0.4607605020,0.4960136926,2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2029-04-27,2030-02-24,0.1573490924,0.4926803272,6.5061151318,2.5415225905,0.5086105105,1.1435796811,0.3738190001,1.2166751869,2.5789118851,...,1.9251131398,0.9564421643,1.3078706009,2.5855695174,3.0116272425,1.2333392752,0.3510637654,1.9855578853,1.1519286306,2030
2029-04-30,2030-02-25,0.1699177668,0.5253977635,4.9317979614,2.6312044311,0.5303236198,1.1564893479,0.2350726861,1.2295237354,2.7281741747,...,2.0452998150,0.9728011096,1.4903016391,2.9649128092,2.8342414548,1.3288549263,0.9932937055,1.9408029888,1.1994463323,2030
2029-05-03,2030-02-26,0.1593188453,0.5195960201,4.3961674020,2.6153012567,0.5120132122,1.1542000922,0.1898588515,1.2186886325,2.6022979862,...,2.2533609158,1.1258235993,1.5892186272,3.1814689842,2.9754113216,1.3624803029,0.9236682439,1.5250239621,1.2044269238,2030


In [267]:
df = all_transformed.copy()
summer_df = df[df.index.month.isin([12, 1, 2])]
daily_summer_peaks = summer_df.resample('D').max()
daily_summer_peaks = daily_summer_peaks.dropna(axis=0, how='all')
daily_summer_peaks = daily_summer_peaks.reset_index()

daily_summer_peaks['Summer_Year'] = daily_summer_peaks.timestamp.apply(
    lambda x: x.year + 1 if x.month >= 12 else x.year
)




In [272]:
#Save the hydro time series data
hydro_dict = load_hydro_data()
AP_hydro = hydro_dict["MonthlyNaturalInflow_Anthony_Pieman_RefYear4006_StepChange"]
AP_hydro = AP_hydro[AP_hydro.index < '2044-07-02']
aa = pd.DataFrame(AP_hydro).drop_duplicates()
ddi = aa.resample('30min').asfreq()
#aa.loc[aa['index'] == aa.timestamp.max(), 'index'] = last_index
ddi = ddi.interpolate(method='linear')
hydro = pd.DataFrame(ddi[ddi.index < '2044-07-01'])
hydro = hydro.rename(columns={'Inflows':'TAS_hydro'})
hydro['QLD_hydro'] = hydro['TAS_hydro']
hydro['NSW_hydro'] = hydro['TAS_hydro']
hydro['VIC_hydro'] = hydro['TAS_hydro']
hydro['SA_hydro'] = hydro['TAS_hydro']
hydro.index.name = 'timestamp'
hydro.to_csv(os.path.join(odir, "2024_ISP_SC_hydro_20yrs.csv"))

In [309]:
hydro_ed = pd.DataFrame(AP_hydro[AP_hydro.index < '2030-07-01'])
hydro_ed = hydro_ed.reset_index()
summer_hydro = hydro_ed[hydro_ed.timestamp.dt.month.isin([12, 1, 2])]
summer_hydro['Summer_Year'] = summer_hydro.timestamp.apply(
    lambda x: x.year + 1 if x.month >= 12 else x.year
)

daily_summer_hydro = summer_hydro.dropna(axis=0, how='all')
new_index = pd.date_range(start=daily_summer_hydro.timestamp.min(), 
                          end=daily_summer_hydro.timestamp.max(), freq='3D')

new_index = new_index[0:daily_summer_hydro.shape[0]]
#new_index = new_index.floo('D')
daily_summer_hydro = daily_summer_hydro.rename(columns={"timestamp": "original_daily_timestamp"})
daily_summer_hydro["timestamp"] = new_index
daily_summer_hydro = daily_summer_hydro.set_index('timestamp')
daily_summer_hydro = daily_summer_hydro.rename(columns={'Inflows':'TAS_hydro'})
daily_summer_hydro['QLD_hydro'] = daily_summer_hydro['TAS_hydro']
daily_summer_hydro['NSW_hydro'] = daily_summer_hydro['TAS_hydro']
daily_summer_hydro['VIC_hydro'] = daily_summer_hydro['TAS_hydro']
daily_summer_hydro['SA_hydro'] = daily_summer_hydro['TAS_hydro']

daily_summer_hydro.to_csv(os.path.join(odir, "summer_daily_peak_hydro_2025_2030.csv"))

In [310]:
daily_summer_hydro

,original_daily_timestamp,TAS_hydro,Summer_Year,QLD_hydro,NSW_hydro,VIC_hydro,SA_hydro
timestamp,,,,,,,
2024-12-01,2024-12-01,23.1496392717,2025,23.1496392717,23.1496392717,23.1496392717,23.1496392717
2024-12-04,2024-12-02,23.1496392717,2025,23.1496392717,23.1496392717,23.1496392717,23.1496392717
2024-12-07,2024-12-03,23.1496392717,2025,23.1496392717,23.1496392717,23.1496392717,23.1496392717
2024-12-10,2024-12-04,23.1496392717,2025,23.1496392717,23.1496392717,23.1496392717,23.1496392717
2024-12-13,2024-12-05,23.1496392717,2025,23.1496392717,23.1496392717,23.1496392717,23.1496392717
...,...,...,...,...,...,...,...
2029-04-27,2030-02-24,6.6766394304,2030,6.6766394304,6.6766394304,6.6766394304,6.6766394304
2029-04-30,2030-02-25,6.6766394304,2030,6.6766394304,6.6766394304,6.6766394304,6.6766394304
2029-05-03,2030-02-26,6.6766394304,2030,6.6766394304,6.6766394304,6.6766394304,6.6766394304
